In [3]:
import pandas as pd
import numpy as np

# MGI_DO

In [4]:
BASE_PATH    = "/storage/Arushi/090526_EvoAge/kg_formation/"

In [44]:
NCBI_MOUSE_gene = pd.read_csv(f'{BASE_PATH}data_collection/databases_for_mapping/ncbi/Mus_musculus.gene_info',sep = '\t')
NCBI_MOUSE_gene

# Extract ENSMUSG pattern using regex and save in new column
NCBI_MOUSE_gene['ENSMBL'] = NCBI_MOUSE_gene['dbXrefs'].str.extract(r'(ENSMUSG\d+)', expand=False)
NCBI_MOUSE_gene
NCBI_mouse_gene_Locus_2_GeneSymd_ict = dict(zip(NCBI_MOUSE_gene['ENSMBL'], NCBI_MOUSE_gene['Symbol']))
NCBI_mouse_gene_Locus_2_GeneSymd_ict
NCBI_mouse_SYM_Descrip_dict = dict(zip(NCBI_MOUSE_gene['Symbol'], NCBI_MOUSE_gene['description']))
NCBI_mouse_SYM_Descrip_dict
# NCBI_MOUSE_gene

{'Pzp': 'PZP, alpha-2-macroglobulin like',
 'Aanat': 'arylalkylamine N-acetyltransferase',
 'Aatk': 'apoptosis-associated tyrosine kinase',
 'Abca1': 'ATP-binding cassette, sub-family A member 1',
 'Abca4': 'ATP-binding cassette, sub-family A member 4',
 'Abca2': 'ATP-binding cassette, sub-family A member 2',
 'Abcb7': 'ATP-binding cassette, sub-family B member 7',
 'Abcg1': 'ATP binding cassette subfamily G member 1',
 'Abi1': 'abl interactor 1',
 'Abl1': 'c-abl oncogene 1, non-receptor tyrosine kinase',
 'Abl2': 'ABL proto-oncogene 2, non-receptor tyrosine kinase',
 'Scgb1b27': 'secretoglobin, family 1B, member 27',
 'ac': 'absent corpus callosum',
 'Acadl': 'acyl-Coenzyme A dehydrogenase, long-chain',
 'Acadm': 'acyl-Coenzyme A dehydrogenase, medium chain',
 'Acadvl': 'acyl-Coenzyme A dehydrogenase, very long chain',
 'Acads': 'acyl-Coenzyme A dehydrogenase, short chain',
 'Slc33a1': 'solute carrier family 33 (acetyl-CoA transporter), member 1',
 'Asic2': 'acid-sensing ion channel 2

In [39]:
MP_ontology2 = pd.read_csv(f'{BASE_PATH}data_collection/mgi_do/ALL_Phenotype.rpt', sep = '\t')
# Explode the phenotypes into separate rows
MP_ontology2 = MP_ontology2.assign(Phenotypes=MP_ontology2["Phenotypes"].str.split("|")).explode("Phenotypes")

# Split each entry into MP_ID and Phenotype
MP_ontology2[["MP_ID", "Phenotype"]] = MP_ontology2["Phenotypes"].str.split(",", n=1, expand=True)

# Drop the original Phenotypes column
MP_ontology2 = MP_ontology2.drop(columns="Phenotypes").reset_index(drop=True)

MP_ontology2
MP_ontology_dict = dict(zip(MP_ontology2['MP_ID'], MP_ontology2['Phenotype']))
MP_ontology_dict

{'MP:0003036': 'vertebral transformation',
 'MP:0000914': 'exencephaly',
 'MP:0011089': 'perinatal lethality, complete penetrance',
 'MP:0000063': 'decreased bone mineral density',
 'MP:0000194': 'increased circulating calcium level',
 'MP:0002965': 'increased circulating serum albumin level',
 'MP:0002968': 'increased circulating alkaline phosphatase level',
 'MP:0005553': 'increased circulating creatinine level',
 'MP:0000849': 'abnormal cerebellum morphology',
 'MP:0000880': 'decreased Purkinje cell number',
 'MP:0001405': 'impaired coordination',
 'MP:0001529': 'abnormal vocalization',
 'MP:0002075': 'abnormal coat/hair pigmentation',
 'MP:0002229': 'neurodegeneration',
 'MP:0003225': 'axonal dystrophy',
 'MP:0003243': 'abnormal dopaminergic neuron morphology',
 'MP:0004077': 'abnormal striatum morphology',
 'MP:0005277': 'abnormal brainstem morphology',
 'MP:0006035': 'abnormal mitochondrial morphology',
 'MP:0006082': 'CNS inflammation',
 'MP:0008493': 'alpha-synuclein inclusion 

In [57]:
# mouse_Gene_ortho_Phenotype_path[3].str[:2].unique()

file = pd.read_csv(f'{BASE_PATH}data_collection/mgi_do/MGI_PhenoGenoMP.rpt', sep = '\t',header = None)

def extract_genes(val):
    parts = str(val).split("|")
    genes = [p.split("<")[0].strip() for p in parts]
    return "|".join(genes)
file
file["head"] = file[1].apply(extract_genes)

file = file.assign(head=file["head"].str.split("|")).explode("head").reset_index(drop=True)

file['tail'] = file[3].map(MP_ontology_dict)
file = file[~file['tail'].isna()]
file#.columns()
file['kg_source'] = 'MGI_DO'

file['tail_detail_name'] = file['tail']
file['head_detail_name'] = file['head'].map(NCBI_mouse_SYM_Descrip_dict)
file = file[~file['head_detail_name'].isna()]

file['head_id_is'] = 'NCBI_ID'
file['tail_id_is'] = ''
file['head_type'] = 'Gene'
file['tail_type'] = 'Phenotype'
file['relation'] = 'Gene_Phenotype'

file['species'] = 'M.musculus'
file['relation_type'] = ''

file = file[[
    "head",
    "relation",
    "tail",
    "head_type",
    "relation_type",
    "tail_type",
    "kg_source",
    "head_id_is",
    "tail_id_is",
    "head_detail_name",
    "tail_detail_name",
    "species"
]]

file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,Rb1,Gene_Phenotype,liver hypoplasia,Gene,,Phenotype,MGI_DO,NCBI_ID,,RB transcriptional corepressor 1,liver hypoplasia,M.musculus
1,Rb1,Gene_Phenotype,abnormal placenta labyrinth morphology,Gene,,Phenotype,MGI_DO,NCBI_ID,,RB transcriptional corepressor 1,abnormal placenta labyrinth morphology,M.musculus
2,Rb1,Gene_Phenotype,decreased embryo size,Gene,,Phenotype,MGI_DO,NCBI_ID,,RB transcriptional corepressor 1,decreased embryo size,M.musculus
3,Rb1,Gene_Phenotype,abnormal trigeminal ganglion morphology,Gene,,Phenotype,MGI_DO,NCBI_ID,,RB transcriptional corepressor 1,abnormal trigeminal ganglion morphology,M.musculus
4,Rb1,Gene_Phenotype,abnormal dorsal root ganglion morphology,Gene,,Phenotype,MGI_DO,NCBI_ID,,RB transcriptional corepressor 1,abnormal dorsal root ganglion morphology,M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...
556456,Casp8,Gene_Phenotype,decreased T cell number,Gene,,Phenotype,MGI_DO,NCBI_ID,,caspase 8,decreased T cell number,M.musculus
556457,Ripk1,Gene_Phenotype,decreased T cell number,Gene,,Phenotype,MGI_DO,NCBI_ID,,receptor (TNFRSF)-interacting serine-threonine...,decreased T cell number,M.musculus
556458,Ripk3,Gene_Phenotype,decreased mature B cell number,Gene,,Phenotype,MGI_DO,NCBI_ID,,receptor-interacting serine-threonine kinase 3,decreased mature B cell number,M.musculus
556459,Casp8,Gene_Phenotype,decreased mature B cell number,Gene,,Phenotype,MGI_DO,NCBI_ID,,caspase 8,decreased mature B cell number,M.musculus


In [58]:
set(file['relation']),set(file['head_type']),set(file['tail_type']),set(file['kg_source']),set(file['head_id_is']),set(file['tail_id_is'])

({'Gene_Phenotype'}, {'Gene'}, {'Phenotype'}, {'MGI_DO'}, {'NCBI_ID'}, {''})

In [59]:
# Count true NaN values
true_nan_count = file.isna().sum()

# Count string 'NAN' values (case-insensitive)
string_nan_count = file.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

# Combine both counts
nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})
nan_summary

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [61]:
# First, concatenate all dataframes as before (assuming `final_df` already exists from earlier step)

# Group by the 3 key columns
group_cols = ['head', 'relation', 'tail']

# For kg_source, combine with '::'
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

# Merge duplicates
final_df_dedup = file.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,1110059G10Rik,Gene_Phenotype,vertebral transformation,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1110059G10 gene,vertebral transformation
1,1500009L16Rik,Gene_Phenotype,decreased bone mineral density,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1500009L16 gene,decreased bone mineral density
2,1500009L16Rik,Gene_Phenotype,increased circulating alkaline phosphatase level,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1500009L16 gene,increased circulating alkaline phosphatase level
3,1500009L16Rik,Gene_Phenotype,increased circulating aspartate transaminase l...,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1500009L16 gene,increased circulating aspartate transaminase l...
4,1500009L16Rik,Gene_Phenotype,increased circulating calcium level,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1500009L16 gene,increased circulating calcium level
...,...,...,...,...,...,...,...,...,...,...,...
282883,wn,Gene_Phenotype,hypopigmentation,Gene,,Phenotype,MGI_DO,NCBI_ID,,white nose,hypopigmentation
282884,wn,Gene_Phenotype,irregular coat pigmentation,Gene,,Phenotype,MGI_DO,NCBI_ID,,white nose,irregular coat pigmentation
282885,wn,Gene_Phenotype,variable depigmentation,Gene,,Phenotype,MGI_DO,NCBI_ID,,white nose,variable depigmentation
282886,wrmod1,Gene_Phenotype,gliosis,Gene,,Phenotype,MGI_DO,NCBI_ID,,wobbler modifier 1,gliosis


In [63]:
final_df_dedup.to_csv(f'{BASE_PATH}processed_data/mgi_do/mgido_MOUSE_GENE_PHENOTYPE.csv',index = None)

# HMD_HP

In [79]:
df = pd.read_csv(f'{BASE_PATH}data_collection/hmd_hp/HMD_HumanPhenotype.rpt', sep = '\t',header = None)
df_new = df[~df[4].isna()]
df_new['head'] = df_new[2]
df_new['tail'] = df_new[4]

# Split tail values by ', '
df_new['tail'] = df_new['tail'].str.split(', ')

# Explode into separate rows
df_new = df_new.explode('tail')

# Optional: reset index
df_new = df_new.reset_index(drop=True)
df_new

/tmp/ipykernel_1726764/1960728655.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['head'] = df_new[2]
/tmp/ipykernel_1726764/1960728655.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['tail'] = df_new[4]
/tmp/ipykernel_1726764/1960728655.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/ind

,0,1,2,3,4,5,head,tail
0,A1CF,29974,A1cf,MGI:1917115,"MP:0005367, MP:0005369, MP:0005370, MP:0005376...",NaN,A1cf,MP:0005367
1,A1CF,29974,A1cf,MGI:1917115,"MP:0005367, MP:0005369, MP:0005370, MP:0005376...",NaN,A1cf,MP:0005369
2,A1CF,29974,A1cf,MGI:1917115,"MP:0005367, MP:0005369, MP:0005370, MP:0005376...",NaN,A1cf,MP:0005370
3,A1CF,29974,A1cf,MGI:1917115,"MP:0005367, MP:0005369, MP:0005370, MP:0005376...",NaN,A1cf,MP:0005376
4,A1CF,29974,A1cf,MGI:1917115,"MP:0005367, MP:0005369, MP:0005370, MP:0005376...",NaN,A1cf,MP:0005378
...,...,...,...,...,...,...,...,...
81209,ZZEF1,23140,Zzef1,MGI:2444286,"MP:0003631, MP:0005367, MP:0005376, MP:0005378...",NaN,Zzef1,MP:0005378
81210,ZZEF1,23140,Zzef1,MGI:2444286,"MP:0003631, MP:0005367, MP:0005376, MP:0005378...",NaN,Zzef1,MP:0005386
81211,ZZEF1,23140,Zzef1,MGI:2444286,"MP:0003631, MP:0005367, MP:0005376, MP:0005378...",NaN,Zzef1,MP:0005390
81212,ZZEF1,23140,Zzef1,MGI:2444286,"MP:0003631, MP:0005367, MP:0005376, MP:0005378...",NaN,Zzef1,MP:0005391


In [80]:
df_new['tail_detail_name'] = df_new['tail'].map(MP_ontology_dict)
df_new = df_new[~df_new['tail_detail_name'].isna()]
df_new#.columns()
df_new['kg_source'] = 'MGI_DO'
df_new
df_new['tail'] = df_new['tail_detail_name']
df_new['head_detail_name'] = df_new['head'].map(NCBI_mouse_SYM_Descrip_dict)
df_new = df_new[~df_new['head_detail_name'].isna()]

df_new['head_id_is'] = 'NCBI_ID'
df_new['tail_id_is'] = ''
df_new['head_type'] = 'Gene'
df_new['tail_type'] = 'Phenotype'
df_new['relation'] = 'Gene_Phenotype'

df_new['species'] = 'M.musculus'
df_new['relation_type'] = ''

df_new = df_new[[
    "head",
    "relation",
    "tail",
    "head_type",
    "relation_type",
    "tail_type",
    "kg_source",
    "head_id_is",
    "tail_id_is",
    "head_detail_name",
    "tail_detail_name",
    "species"
]]

df_new

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,A1cf,Gene_Phenotype,renal/urinary system phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,APOBEC1 complementation factor,renal/urinary system phenotype,M.musculus
1,A1cf,Gene_Phenotype,muscle phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,APOBEC1 complementation factor,muscle phenotype,M.musculus
2,A1cf,Gene_Phenotype,liver/biliary system phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,APOBEC1 complementation factor,liver/biliary system phenotype,M.musculus
3,A1cf,Gene_Phenotype,homeostasis/metabolism phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,APOBEC1 complementation factor,homeostasis/metabolism phenotype,M.musculus
4,A1cf,Gene_Phenotype,growth/size/body region phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,APOBEC1 complementation factor,growth/size/body region phenotype,M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...
81209,Zzef1,Gene_Phenotype,growth/size/body region phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,"zinc finger, ZZ-type with EF hand domain 1",growth/size/body region phenotype,M.musculus
81210,Zzef1,Gene_Phenotype,behavior/neurological phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,"zinc finger, ZZ-type with EF hand domain 1",behavior/neurological phenotype,M.musculus
81211,Zzef1,Gene_Phenotype,skeleton phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,"zinc finger, ZZ-type with EF hand domain 1",skeleton phenotype,M.musculus
81212,Zzef1,Gene_Phenotype,vision/eye phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,"zinc finger, ZZ-type with EF hand domain 1",vision/eye phenotype,M.musculus


In [81]:
set(df_new['relation']),set(df_new['head_type']),set(df_new['tail_type']),set(df_new['kg_source']),set(df_new['head_id_is']),set(df_new['tail_id_is'])

({'Gene_Phenotype'}, {'Gene'}, {'Phenotype'}, {'MGI_DO'}, {'NCBI_ID'}, {''})

In [82]:
# Count true NaN values
true_nan_count = df_new.isna().sum()

# Count string 'NAN' values (case-insensitive)
string_nan_count = df_new.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

# Combine both counts
nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})
nan_summary

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [83]:
# First, concatenate all dataframes as before (assuming `final_df` already exists from earlier step)

# Group by the 3 key columns
group_cols = ['head', 'relation', 'tail']

# For kg_source, combine with '::'
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

# Merge duplicates
final_df_dedup = df_new.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,1110059G10Rik,Gene_Phenotype,skeleton phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1110059G10 gene,skeleton phenotype
1,1500009L16Rik,Gene_Phenotype,homeostasis/metabolism phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1500009L16 gene,homeostasis/metabolism phenotype
2,1500009L16Rik,Gene_Phenotype,skeleton phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1500009L16 gene,skeleton phenotype
3,1700001O22Rik,Gene_Phenotype,cardiovascular system phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1700001O22 gene,cardiovascular system phenotype
4,1700001O22Rik,Gene_Phenotype,nervous system phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,RIKEN cDNA 1700001O22 gene,nervous system phenotype
...,...,...,...,...,...,...,...,...,...,...,...
76336,mt-Rnr2,Gene_Phenotype,cardiovascular system phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,16S ribosomal RNA,cardiovascular system phenotype
76337,mt-Rnr2,Gene_Phenotype,growth/size/body region phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,16S ribosomal RNA,growth/size/body region phenotype
76338,mt-Rnr2,Gene_Phenotype,integument phenotype,Gene,,Phenotype,MGI_DO,NCBI_ID,,16S ribosomal RNA,integument phenotype
76339,mt-Rnr2,Gene_Phenotype,mortality/aging,Gene,,Phenotype,MGI_DO,NCBI_ID,,16S ribosomal RNA,mortality/aging


In [84]:
final_df_dedup.to_csv(f'{BASE_PATH}processed_data/hmdhp/hmdhp_MOUSE_GENE_PHENOTYPE.csv',index = None)